# 02c. Финальный отбор колонок → датасеты для EDA

Берём `left/inner` join из `02_preprocessing`, оставляем только нужные колонки
(переименование `_price` → без суффикса, coalesce гео, отбор `keep`) и сохраняем
готовые датасеты, на которых дальше крутятся `03_eda_projects` и прочие графики.


In [1]:
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd if (_cwd / "data").is_dir() else _cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import (
    JOINED_LEFT_PARQUET, JOINED_INNER_PARQUET,
    EDA_LEFT_PARQUET, EDA_INNER_PARQUET, PROCESSED_DIR,
)

pd.set_option("display.max_columns", 60)


In [2]:
RENAME = {
    "project_id_price":    "project_id",
    "building_id_price":   "building_id",
    "project_name_price":  "project_name",
    "developer_price":     "developer",
    "project_class_price": "project_class",
    "region_price":        "region",
    "district_price":      "district",
    "macro_district_price":"macro_district",
    "sales_start_price":   "sales_start",
    "premises_type_price": "premises_type",
    "section_price":       "section",
    "floor_price":         "floor",
    "room_count_price":    "room_count",
    "area_price":          "area",
    "price_price":         "price",
    "price_list_price":    "price_no_discount",
    "price_list_deals":    "price_deals_no_discount",
}

KEEP = [
    "unit_match_key",
    "project_id", "building_id", "project_name", "developer",
    "project_class", "region", "macro_district", "district", "sales_start", "completion_k",
    "premises_type", "section", "floor", "finish_in_report", "unit_typology",
    "room_count", "area",
    "price", "price_no_discount", "discount_pct",
    "finish_tier", "finish_text", "contract_type_k", "stage_k",
    "exposure", "file_date", "lat", "lng",
    "price_deals", "price_deals_no_discount",
    "registration_date_raw", "pledge_type", "mortgage",
]

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# LEFT JOIN
left = pd.read_parquet(JOINED_LEFT_PARQUET)
left = left.rename(columns={k: v for k, v in RENAME.items() if k in left.columns})

for coord in ("lat", "lng"):
    p = left[f"{coord}_price"] if f"{coord}_price" in left.columns else pd.Series(float("nan"), index=left.index)
    d = left[f"{coord}_deals"] if f"{coord}_deals" in left.columns else pd.Series(float("nan"), index=left.index)
    left[coord] = p.where(p.notna(), d)

left = left[[c for c in KEEP if c in left.columns]]

left["completion_k"] = pd.to_datetime(left["completion_k"], dayfirst=True, errors="coerce")
for col in ("price", "price_no_discount", "discount_pct", "price_deals_no_discount"):
    left[col] = pd.to_numeric(left[col].astype(str).str.replace(",", ".", regex=False), errors="coerce").astype("float64")

# дедуп: один снапшот на (лот, дата) — убираем дубли от перекрытия прайс-срезов
left = left.drop_duplicates(["unit_match_key", "file_date"], keep="last")
left = left.sort_values(["unit_match_key", "file_date"]).reset_index(drop=True)

# таргеты: is_sold* = 1 только на ПОСЛЕДНЕЙ строке лота; is_sold_past — по строкам
is_last = ~left["unit_match_key"].duplicated(keep="last")
delta = (left["registration_date_raw"] - left["file_date"]).dt.days
left["is_sold_past"]         = (delta < 0).astype("int8")
left["is_sold"]              = (is_last & (delta >= 0)).astype("int8")
left["is_sold_next_month"]   = (is_last & (delta > 0) & (delta <= 30)).astype("int8")
left["is_sold_next_quarter"] = (is_last & (delta > 0) & (delta <= 90)).astype("int8")

obj = left.select_dtypes("object").columns
if len(obj):
    left[obj] = left[obj].astype("string")

left.to_parquet(EDA_LEFT_PARQUET, index=False)

/var/folders/4d/y7_mlhmn0kq7bgf3mkkj_hxr0000gn/T/ipykernel_8526/1050565521.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj = left.select_dtypes("object").columns


In [ ]:
# INNER JOIN
inner = pd.read_parquet(JOINED_INNER_PARQUET)
inner = inner.rename(columns={k: v for k, v in RENAME.items() if k in inner.columns})

for coord in ("lat", "lng"):
    p = inner[f"{coord}_price"] if f"{coord}_price" in inner.columns else pd.Series(float("nan"), index=inner.index)
    d = inner[f"{coord}_deals"] if f"{coord}_deals" in inner.columns else pd.Series(float("nan"), index=inner.index)
    inner[coord] = p.where(p.notna(), d)

inner = inner[[c for c in KEEP if c in inner.columns]]

inner["completion_k"] = pd.to_datetime(inner["completion_k"], dayfirst=True, errors="coerce")
for col in ("price", "price_no_discount", "discount_pct", "price_deals_no_discount"):
    inner[col] = pd.to_numeric(inner[col].astype(str).str.replace(",", ".", regex=False), errors="coerce").astype("float64")

# дедуп: один снапшот на (лот, дата)
inner = inner.drop_duplicates(["unit_match_key", "file_date"], keep="last")
inner = inner.sort_values(["unit_match_key", "file_date"]).reset_index(drop=True)

is_last = ~inner["unit_match_key"].duplicated(keep="last")
delta = (inner["registration_date_raw"] - inner["file_date"]).dt.days
inner["is_sold_past"]         = (delta < 0).astype("int8")
inner["is_sold"]              = (delta > 0).astype("int8")
inner["is_sold_next_month"]   = ((delta > 0) & (delta <= 30)).astype("int8")
inner["is_sold_next_quarter"] = ((delta > 0) & (delta <= 90)).astype("int8")

obj = inner.select_dtypes("object").columns
if len(obj):
    inner[obj] = inner[obj].astype("string")

inner.to_parquet(EDA_INNER_PARQUET, index=False)

/var/folders/4d/y7_mlhmn0kq7bgf3mkkj_hxr0000gn/T/ipykernel_8526/2320285204.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj = inner.select_dtypes("object").columns


In [5]:
# Отчёт
print(f"left  = {left.shape[0]:>9,} x {left.shape[1]}  ->  {EDA_LEFT_PARQUET.name}")
print(f"inner = {inner.shape[0]:>9,} x {inner.shape[1]}  ->  {EDA_INNER_PARQUET.name}")

print("\nДоля положительных по таргетам (left):")
for t in ("is_sold", "is_sold_past", "is_sold_next_month", "is_sold_next_quarter"):
    print(f"  {t:<22s} {left[t].mean()*100:5.2f}%  ({int(left[t].sum()):,})")

print("\nТипы и пропуски (left):")
for c in left.columns:
    print(f"  {c:<26s} {str(left[c].dtype):<16s} NaN={left[c].isna().mean()*100:.1f}%")

left  = 4,631,451 x 36  ->  eda_left.parquet
inner =   909,364 x 36  ->  eda_inner.parquet

Доля положительных по таргетам (left):
  is_sold                 3.84%  (177,672)
  is_sold_past            0.66%  (30,343)
  is_sold_next_month      1.67%  (77,426)
  is_sold_next_quarter    3.40%  (157,697)

Типы и пропуски (left):
  unit_match_key             string           NaN=0.0%
  project_id                 Int64            NaN=0.0%
  building_id                string           NaN=0.0%
  project_name               string           NaN=0.0%
  developer                  string           NaN=0.0%
  project_class              string           NaN=0.0%
  region                     string           NaN=0.0%
  sales_start                datetime64[us]   NaN=0.3%
  completion_k               datetime64[us]   NaN=0.0%
  premises_type              string           NaN=0.0%
  section                    string           NaN=26.2%
  floor                      Int64            NaN=0.0%
  finish_in_r